In [1]:
# !pip install fusion_solar_py -q

In [2]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")

In [8]:
from dotenv import load_dotenv
import os

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")

In [9]:
import pandas as pd
from energymanagementrl.fusion_solar_extension import FusionSolarClientParsed

In [10]:
# log into the API - with proper credentials...
client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                 huawei_subdomain="uni004eu5")
_plant_id = client.get_plant_ids()[0]
_battery_id = client.get_battery_ids(_plant_id)[0]

In [11]:
# client.set_battery_working_mode(_battery_id, client.BatteryWorkingMode.FULLY_FEED_TO_GRID)
client.set_battery_working_mode(_battery_id, client.BatteryWorkingMode.MAXIMUM_SELF_CONSUMPTION)

In [7]:
plant_data = client.get_plant_stats(_plant_id)

In [8]:
client.get_battery_day_stats(_battery_id)

{'30005': {'pmDataList': [{'counterId': 30005,
    'counterValue': -0.31,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime': 1734044400,
    'timeZoneOffset': 60},
   {'counterId': 30005,
    'counterValue': -0.303,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime': 1734044700,
    'timeZoneOffset': 60},
   {'counterId': 30005,
    'counterValue': -0.305,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime': 1734045000,
    'timeZoneOffset': 60},
   {'counterId': 30005,
    'counterValue': -0.151,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime': 1734045300,
    'timeZoneOffset': 60},
   {'counterId': 30005,
    'counterValue': -0.151,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime': 1734045600,
    'timeZoneOffset': 60},
   {'counterId': 30005,
    'counterValue': -0.148,
    'dnId': 110232156,
    'dstOffset': 0,
    'period': 300,
    'startTime

In [4]:
current_time = pd.Timestamp.now()
grid_connection_time = pd.to_datetime(client.get_plant_details(_plant_id)['gridConnectedTime'].split()[0])
TIMEZONE_LOCAL = 'Europe/Rome'

unformatted_plant_history, final_plant_history=client.get_plant_history(grid_connection_time, current_time,_battery_id,_plant_id,TIMEZONE_LOCAL)

In [6]:
# Display the resulting dataframe
final_plant_history.info()
final_plant_history

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 74742 entries, 2024-03-29 23:00:00 to 2024-12-14 10:25:00
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   production_power_kw  74594 non-null  float16
 1   load_power_kw        74575 non-null  float16
 2   grid_power_kw        74575 non-null  float16
 3   stored_power_kw      74594 non-null  float16
 4   SOC                  74594 non-null  float16
dtypes: float16(5)
memory usage: 1.3 MB


/home/emanuele/IdeaProjects/EnergyManagementRL/.venv/lib/python3.10/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,production_power_kw,load_power_kw,grid_power_kw,stored_power_kw,SOC
timestamp,,,,,
2024-03-29 23:00:00,0.000000,-0.281982,0.000000,0.281982,23.0
2024-03-29 23:05:00,0.000000,-0.270996,-0.020020,0.250977,22.0
2024-03-29 23:10:00,0.000000,-0.273926,-0.010010,0.263916,22.0
2024-03-29 23:15:00,0.000000,-0.268066,-0.010010,0.258057,22.0
2024-03-29 23:20:00,0.000000,-0.267090,-0.010010,0.257080,22.0
...,...,...,...,...,...
2024-12-14 10:05:00,4.699219,-4.285156,0.906250,0.493896,45.0
2024-12-14 10:10:00,4.746094,-4.351562,1.312500,0.916016,44.0
2024-12-14 10:15:00,4.878906,-2.478516,0.810547,-1.590820,44.0


In [7]:
data_folder = '../data/'
unformatted_plant_history.to_csv(data_folder + 'plant_hist_unformatted.csv')
final_plant_history.to_csv(data_folder + 'plant_hist.csv')